# Setup `mailroom-ia` — pas à pas

Ce notebook **déploie le projet de A à Z** dans ton Resource Group. Tu exécutes les cellules une par une, en lisant les explications avant chaque cellule.

## Vue d'ensemble (5 min de lecture)

L'application `mailroom-ia` reçoit des PDFs scannés et les classe automatiquement avec l'IA. Deux containers tournent dans Azure :

- **`web`** — site Next.js où l'admin upload les PDFs. Le PDF est posé dans un **Blob Storage**, puis un message est mis dans une **Queue**.
- **`worker`** — script Python déclenché par la queue (KEDA). Il télécharge le PDF, appelle **Document Intelligence** pour l'OCR, puis **Foundry / gpt-5-mini** pour comprendre de quoi parle le document, et enregistre le résultat dans **Cosmos DB**.

Schéma complet : ouvre [docs/architecture.drawio](docs/architecture.drawio) dans VS Code (extension *Draw.io Integration*) ou sur https://app.diagrams.net.

## Ce qu'on va faire dans ce notebook

| § | Étape | Outil | Durée |
|---|-------|-------|------:|
| 1 | Vérifier les outils installés | `az`, `docker`, `bicep` | 1 min |
| 2 | Se connecter à Azure | `az login` | 1 min |
| 3 | Définir les variables (1 seule à éditer) | Python | 1 min |
| 4 | Déployer l'infra **de base** (Bicep #1) | LAW, App Insights, ACR, Storage, Cosmos, Foundry, ACA Env | 8 min |
| 5 | Créer le projet Foundry | `az cli` | 30 s |
| 6 | Builder + pousser les images Docker | ACR Tasks (cloud build) | 6 min |
| 7 | Déployer les **apps** (Bicep #2) | web + worker + RBAC | 1 min |
| 8 | Tester un upload de bout en bout | curl | 3 min |
| 9 | Nettoyage ciblé (optionnel) | `az` | — |

⏱️ **Total : ~25 min** la première fois. Les redéploiements suivants : ~2-3 min.

## Réseau & sécurité — note importante

**Pour ce workshop, toutes les ressources sont en accès public (`publicNetworkAccess: Enabled`).** Pas de VNet, pas de Private Endpoint. C'est volontairement simple : tu te concentres sur l'IA et le code, pas sur la plomberie réseau.

> 🛡️ Le tenant a une **policy** qui force `publicNetworkAccess: Disabled` sur les services de données (Storage, Cosmos, Foundry) **sauf** si la ressource porte le tag `SecurityControl=ignore`. Ce tag est déjà mis automatiquement par nos templates Bicep — tu n'as rien à faire.
>
> Dans une vraie prod, on remplacerait ça par des **Private Endpoints** + une **VNet** + un **Application Gateway** devant le `web`. Hors scope ici.

## RBAC — qui a le droit de faire quoi

Chaque container a sa propre **Managed Identity** (UAMI = User-Assigned Managed Identity). Pas de mot de passe, pas de clé, Azure gère les tokens.

| Identité | Rôles | Sur quoi |
|----------|-------|----------|
| `uami-web-<id>` | AcrPull | ACR (puller l'image) |
| | Storage Blob Data Contributor | Storage (écrire le PDF) |
| | Storage Queue Data Sender | Storage (poster un message) |
| | Cosmos DB Built-in Data Contributor | Cosmos (lire/écrire documents) |
| `uami-worker-<id>` | AcrPull | ACR |
| | Storage Blob Data Contributor | Storage (télécharger + déplacer) |
| | Storage Queue Data Reader + Message Processor | Storage (KEDA + consommer messages) |
| | Cognitive Services User + OpenAI User | Foundry (appeler DI + LLM) |
| | Cosmos DB Built-in Data Contributor | Cosmos |

Tout est défini dans [infra/infra-apps.bicep](infra/infra-apps.bicep) — pas besoin de l'exécuter à la main.

> ⚠️ **Pré-requis** : avoir suivi les notebooks 01 → 07 du parcours. En particulier comprendre Docker / Container Apps (notebook 07).

> 💰 Coût en idle : ~5-10 €/mois (1 replica web min). Worker = 0 € quand la queue est vide. Nettoyage complet = 0 €.

## 0. Les services Azure utilisés — lecture obligatoire

Avant de cliquer sur quoi que ce soit, prends 5 minutes pour comprendre **ce que fait chaque service**. Ça t'évitera de te perdre quand tu liras les Bicep et le code.

### Compute (là où le code tourne)

#### **Azure Container Apps (ACA)**
Le Kubernetes "managed-managed" d'Azure : tu donnes une image Docker, Azure s'occupe du cluster, du scaling, du HTTPS, des rolling updates. Pas besoin de comprendre K8s.

- **Container App** → service long-running (notre `web`). 1 à N replicas, ingress HTTPS.
- **Container App Job** → tâche ponctuelle (notre `worker`). Tourne, finit, s'éteint.
- **Container App Environment** → le "cluster" qui héberge tes apps. Une env peut héberger plusieurs apps qui partagent réseau et logs.

#### **KEDA (Kubernetes Event-Driven Autoscaling)**
Le moteur de scaling d'ACA, embarqué automatiquement. Il regarde une **source d'événements** (queue, topic, cron…) et décide combien de replicas/jobs lancer.

Dans notre cas : **scaler `azure-queue`** → KEDA poll la Storage Queue toutes les 30 s. Dès qu'il y a >= 1 message, il déclenche une exécution du Job `worker`. Quand la queue est vide, plus aucun replica → **zéro coût de compute**.

> KEDA ≠ Azure Functions. Functions est plus simple mais moins flexible (pas de Docker custom, runtimes limités). Pour un worker Python custom avec deps lourdes, ACA Job + KEDA est mieux.

#### **Azure Container Registry (ACR)**
Un registre Docker privé dans ton tenant. ACA tire les images d'ici (jamais Docker Hub en prod). On l'utilise aussi pour **builder les images dans le cloud** avec `az acr build` — plus besoin de Docker actif en local.

### Stockage (là où les données vivent)

#### **Azure Storage Account**
Un compte de stockage = 4 services en un :

| Sous-service | Notre usage |
|--------------|-------------|
| **Blob** | Stocker les PDFs uploadés (`mailroom/_inbox/<uuid>.pdf`, puis rangés dans `mailroom/clients/<clientId>/...`) |
| **Queue** | File d'attente FIFO simple (notre `doc-to-classify`). 64 KB max par message, donc on y met juste un JSON `{id, blobName, mimeType, size}` — pas le PDF |
| Table | (pas utilisé) |
| File | (pas utilisé) |

> 🤔 **Pourquoi Storage Queue et pas Service Bus ou Event Grid ?**
>
> - **Storage Queue** : ultra simple, gratuit, ~1 ms de latence, parfait pour un POC ou un volume modéré. C'est ce qu'on utilise.
> - **Service Bus** : queue "entreprise" — dead-letter intégré, sessions, transactions, pub/sub. Plus cher mais plus robuste. À utiliser quand tu veux du replay garanti ou plusieurs consommateurs.
> - **Event Grid** : pas une queue, c'est un **broker pub/sub d'événements** ("un blob a été créé", "une ressource a changé"). Très utile si tu veux réagir à des événements Azure natifs (ex. "un fichier déposé dans tel blob déclenche tel agent"). Dans notre cas, c'est `web` qui pousse explicitement le message → pas besoin d'Event Grid. Si on voulait que la queue se remplisse automatiquement quand un blob arrive (ex. drop d'un fournisseur externe), on brancherait Event Grid → Storage Queue dessus.

#### **Azure Cosmos DB (NoSQL serverless)**
Base de données JSON managed. **Serverless** = tu payes par requête (par RU — Request Unit), pas à l'heure. Idéal pour un POC avec trafic faible/imprevisible.

Deux containers :
- `documents` (partition `/clientId`) : 1 entrée par PDF classé.
- `clients` (partition `/id`) : 1 entrée par client connu (utilisé par le LLM pour matcher le destinataire).

> 🔐 Cosmos a **son propre système de RBAC** (les `sqlRoleAssignments`), séparé d'Azure RBAC. Pour qu'un UAMI puisse lire/écrire les items, on lui assigne le rôle built-in **`Cosmos DB Built-in Data Contributor`** (étape §7.5).

### Intelligence Artificielle

#### **Microsoft Foundry**
La plateforme IA Azure (anciennement Azure AI Studio). Une ressource Foundry de type `AIServices` regroupe **plusieurs services IA derrière un seul endpoint** :

- **Document Intelligence** (anciennement Form Recognizer) — OCR + extraction structurée (`prebuilt-layout`, `prebuilt-invoice`, etc.).
- **Azure OpenAI** — déploiement de modèles (GPT, embeddings). On déploie ici `gpt-5-mini`.
- **Foundry Agents, Evaluations, Tracing** — hors scope de ce notebook.

Un compte Foundry héberge plusieurs **projets** (=workspaces) indépendants.

### Observabilité

#### **Log Analytics Workspace (LAW)**
Le data warehouse de logs/métriques d'Azure. Les Container Apps y poussent automatiquement leurs `stdout/stderr` (table `ContainerAppConsoleLogs_CL`) et les événements système (`ContainerAppSystemLogs_CL`). On l'interroge en **KQL** (Kusto Query Language).

#### **Application Insights**
Le SDK de tracing applicatif Azure (compatible OpenTelemetry). Le worker enverra ses spans ("j'ai appelé DI, ça a pris 800 ms", "j'ai callé le LLM, 12 s") → utile pour le notebook 06 (monitoring + evaluation).

### Identités & sécurité

#### **Managed Identity (UAMI / SAMI)**
Une "identité Azure sans mot de passe" pour tes ressources. Tu attaches une identité à un container, et Azure injecte un endpoint local (`IDENTITY_ENDPOINT`) que ton code interroge pour obtenir un token Entra ID. Le SDK Azure (`DefaultAzureCredential`) fait ça transparently.

- **System-Assigned** : créée et détruite avec la ressource. Simple.
- **User-Assigned** : ressource indépendante, partageable. **C'est ce qu'on utilise** car les rôles `AcrPull` doivent exister AVANT que le container ne démarre.

#### **Azure Policy**
Des règles appendées à la subscription qui **forcent** une config. Ex. notre tenant a une policy qui désactive `publicNetworkAccess` sur les services de données — d'où le tag `SecurityControl=ignore` qu'on met partout dans nos Bicep pour l'exempter (workshop only).

### En résumé, le chemin d'un PDF

```
[User] → web (ACA) → Blob + Queue (Storage) → KEDA détecte → worker (ACA Job)
                                                                  ↓
                                              Document Intelligence + Foundry LLM
                                                                  ↓
                                                              Cosmos DB
                                                                  ↓
                                          Logs/métriques → LAW + App Insights
```

⚠️ **Réseau** : pour ce workshop, tous ces services sont en **accès public** (avec leur firewall ouvert). Pas de VNet, pas de Private Endpoint. En prod on isolerait tout ça dans une VNet, mais c'est hors scope ici.

## 1. Vérifier que les outils sont installés

On a besoin de trois outils :

- **Azure CLI** (`az`) : pour piloter Azure depuis la ligne de commande.
- **Bicep CLI** : pour déployer l'infrastructure (livré avec az).
- **Docker** : juste pour la commande `docker --version`. **Tu n'as PAS besoin de Docker Desktop actif** — on builde les images dans le cloud avec `az acr build` plus tard.

Si une commande échoue : (re)installe l'outil et relance.

In [ ]:
# Vérification
!az --version
!az bicep version
!docker --version

In [ ]:
# Installer l'extension Azure CLI pour Container Apps (idempotent — ne fait rien si déjà installée).
# Puis enregistrer les "resource providers" : c'est l'opération qui dit à ta subscription
# "j'autorise la création de ressources de tel type". À faire UNE fois par subscription.
!az extension add --name containerapp --upgrade --only-show-errors
!az provider register --namespace Microsoft.App --wait
!az provider register --namespace Microsoft.OperationalInsights --wait
!az provider register --namespace Microsoft.CognitiveServices --wait
!az provider register --namespace Microsoft.DocumentDB --wait
!az provider register --namespace Microsoft.ManagedIdentity --wait

## 2. Se connecter à Azure

Avant TOUTE commande `az` qui touche aux ressources, il faut s'authentifier.


- Sur un poste perso : `az login` ouvre un navigateur.Après connexion, vérifie que tu es sur la **bonne subscription** (celle de ton workshop, pas une perso).

- Dans un Codespace / SSH : `az login --use-device-code` (ça affiche un code à coller sur https://microsoft.com/devicelogin).

In [ ]:
# Décommente la première ligne si pas encore connecté, puis relance.
# !az login --use-device-code

# Si tu as plusieurs subscriptions, choisis-en une :
# !az account set --subscription "<nom-ou-id>"

!az account show --output table

## 3. Définir les variables

✨ **Une seule variable à éditer dans la cellule suivante** : `RG` = le nom de ton Resource Group (format `rg-stage-<ton-identifiant>`).

Tous les autres noms (ACR, Storage, Cosmos, Foundry, etc.) sont **calculés automatiquement** à partir de ton identifiant. Ça garantit que deux stagiaires n'ont pas de collision de noms (les noms de Storage Account et ACR doivent être uniques au monde 👋).

La cellule crée aussi une petite fonction `az()` qui rend les erreurs lisibles — sinon les messages d'`az cli` sont vite illisibles.

In [ ]:
import os, re, subprocess
from pathlib import Path

# 👇 SEULE VARIABLE À ÉDITER
RG = "rg-stage-sadk"

m = re.match(r"^rg-stage-(?P<id>[a-z0-9\-]+)$", RG)
assert m, f"❌ Le RG '{RG}' ne suit pas la convention 'rg-stage-<id>'."
USER_ID = m.group("id")


def az(cmd: str) -> str:
    """Lance une commande az et renvoie stdout. Affiche stderr lisible en cas d'échec."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(
            f"❌ Commande échouée (exit {result.returncode}):\n"
            f"   $ {cmd}\n"
            f"   stderr: {result.stderr.strip() or '(vide)'}\n"
            f"   stdout: {result.stdout.strip() or '(vide)'}\n"
            f"   👉 Vérifiez : (1) vous êtes connecté (`az login`), "
            f"(2) bonne subscription active (`az account show`), "
            f"(3) le RG '{RG}' existe et vous y avez Owner/Contributor."
        )
    return result.stdout.strip()


LOCATION = az(f"az group show --name {RG} --query location -o tsv")

# Chemin racine du projet (où on se trouve)
PROJECT_DIR = Path.cwd()
INFRA_DIR = PROJECT_DIR / "infra"
WEB_DIR = PROJECT_DIR / "web"
WORKER_DIR = PROJECT_DIR / "worker"

# Noms dérivés (alignés sur les Bicep)
clean = USER_ID.replace("-", "")
ACR_NAME = f"acrmailroom{clean}"
ACR_LOGIN = f"{ACR_NAME}.azurecr.io"
STORAGE_NAME = f"stmailroom{clean}"
COSMOS_NAME = f"cosmos-stage-{USER_ID}"
FOUNDRY_NAME = f"aif-stage-{USER_ID}"
FOUNDRY_PROJECT = "mailroom-project"
ACA_ENV = f"aca-stage-{USER_ID}"
WEB_APP = f"web-stage-{USER_ID}"
WORKER_JOB = f"worker-stage-{USER_ID}"

for k, v in {
    "RG": RG, "USER_ID": USER_ID, "LOCATION": LOCATION,
    "ACR_NAME": ACR_NAME, "ACR_LOGIN": ACR_LOGIN, "STORAGE_NAME": STORAGE_NAME,
    "COSMOS_NAME": COSMOS_NAME, "FOUNDRY_NAME": FOUNDRY_NAME,
    "FOUNDRY_PROJECT": FOUNDRY_PROJECT, "ACA_ENV": ACA_ENV,
    "WEB_APP": WEB_APP, "WORKER_JOB": WORKER_JOB,
}.items():
    os.environ[k] = v

print(f"✓ RG trouvé : {RG}  (région : {LOCATION})")
print(f"  Identifiant : {USER_ID}")
print(f"  ACR         : {ACR_LOGIN}")
print(f"  Foundry     : {FOUNDRY_NAME}")
print(f"  ACA env     : {ACA_ENV}")

## 4. Déployer l'infrastructure de base (Bicep #1)

[infra/infra-base.bicep](infra/infra-base.bicep) crée **toutes les ressources "plumbing"** dont on aura besoin avant de pouvoir lancer le code :

| Ressource | À quoi ça sert |
|-----------|----------------|
| Log Analytics Workspace | Stocker tous les logs (web, worker, ACA env) |
| Application Insights | Tracing & métriques applicatives (latérence, exceptions…) |
| Azure Container Registry (ACR) | Héberger nos images Docker `mailroom-web` et `mailroom-worker` |
| Storage Account | Blob pour les PDFs + Queue pour le pipeline async |
| Cosmos DB (serverless) | Métadonnées : 1 document par PDF classé, 1 document par client |

| Microsoft Foundry (kind=AIServices) | LLM (gpt-5-mini) + Document Intelligence en même endpoint |La latérence inter-région West EU ↔ France Central = quelques ms, sans impact pour ce POC.

| Container Apps Environment | L'environnement de runtime managed-K8s où tournent `web` et `worker` |

2. **ACA Environment peut renvoyer `ManagedEnvironmentCapacityHeavyUsageError`** (capacité AKS saturée). Idem — on bascule en France Central si besoin via `acaLocation`.

⏱️ **~8 min** (Cosmos est le plus lent).1. **Cosmos en West Europe peut renvoyer `ServiceUnavailable`** (capacité zonale saturée). On le déploie donc en **France Central** par défaut (param`cosmosLocation`). Le reste reste dans la région du RG.


### ⚠️ Deux pièges connus

In [ ]:
# Choix des régions cibles.
#   ACA_LOCATION    : région du Container Apps Environment.
#   COSMOS_LOCATION : région du compte Cosmos.
# On force West Europe + France Central par défaut car c'est la combinaison qui passe
# de manière fiable au moment du workshop.
ACA_LOCATION = "westeurope"
COSMOS_LOCATION = "francecentral"

os.environ["ACA_LOCATION"] = ACA_LOCATION
os.environ["COSMOS_LOCATION"] = COSMOS_LOCATION
print(f"Région du RG     : {LOCATION}")
print(f"ACA Env          : {ACA_LOCATION}")
print(f"Cosmos DB        : {COSMOS_LOCATION}")

### 4.1 (Optionnel) Nettoyer un Cosmos en échec

Si un déploiement précédent s'est arrêté en plein milieu, Cosmos peut rester bloqué dans l'état **`Failed`**. Dans ce cas, ARM refuse de le recréer tant qu'on ne le supprime pas explicitement. La cellule suivante détecte ce cas.

In [ ]:
# Vérifier si Cosmos est en provisioning failed (cas du retry après crash)
cosmos_state = subprocess.run(
    f"az cosmosdb show --name {COSMOS_NAME} --resource-group {RG} "
    f"--query provisioningState -o tsv",
    shell=True, capture_output=True, text=True,
).stdout.strip()

if cosmos_state:
    print(f"Cosmos {COSMOS_NAME} → state: {cosmos_state}")
    if cosmos_state.lower() == "failed":
        print("⚠️  Cosmos est en état 'Failed' — il faut le supprimer avant de relancer la passe 1.")
        print("    Décommentez la ligne ci-dessous pour le supprimer (~2 min) :")
        print(f"    !az cosmosdb delete --name {COSMOS_NAME} --resource-group {RG} --yes")
        # !az cosmosdb delete --name $COSMOS_NAME --resource-group $RG --yes
else:
    print(f"✓ Cosmos {COSMOS_NAME} n'existe pas encore → tout est OK pour déployer.")

In [ ]:
# Validation Bicep — c'est un dry-run : ARM vérifie la syntaxe et les permissions
# SANS rien créer. Si ça passe ici, le déploiement réel a de bonnes chances de marcher.
!az deployment group validate \
    --resource-group $RG \
    --template-file "{INFRA_DIR}/infra-base.bicep" \
    --parameters userId=$USER_ID acaLocation=$ACA_LOCATION cosmosLocation=$COSMOS_LOCATION \
    --output table

In [ ]:
# Déploiement réel de la passe 1. ~8 min.
# Bicep est idempotent : si tu relances, il ne touche que ce qui a changé.
!az deployment group create \
    --name mailroom-base \
    --resource-group $RG \
    --template-file "{INFRA_DIR}/infra-base.bicep" \
    --parameters userId=$USER_ID acaLocation=$ACA_LOCATION cosmosLocation=$COSMOS_LOCATION \
    --output table

In [ ]:
import json

# 1. État du déploiement
state = az(
    f"az deployment group show --resource-group {RG} --name mailroom-base "
    f"--query properties.provisioningState -o tsv"
)
print(f"Déploiement mailroom-base : provisioningState = {state}")

if state.lower() != "succeeded":
    print("\n❌ Le déploiement n'est pas en 'Succeeded' → pas d'outputs à récupérer.")
    print("Détails des opérations en échec :\n")
    # On passe par subprocess (pas !... shell) pour interpoler {RG} correctement
    fails = subprocess.run(
        [
            "az", "deployment", "operation", "group", "list",
            "--name", "mailroom-base",
            "--resource-group", RG,
            "--query",
            "[?properties.provisioningState=='Failed']."
            "{resource:properties.targetResource.resourceName,"
            "type:properties.targetResource.resourceType,"
            "code:properties.statusMessage.error.code,"
            "message:properties.statusMessage.error.message}",
            "-o", "jsonc",
        ],
        shell=True, capture_output=True, text=True,
    )
    print(fails.stdout or fails.stderr)
    raise RuntimeError(
        f"Déploiement en état '{state}'. Voir les détails ci-dessus, "
        f"corriger et relancer la cellule de déploiement."
    )

# 2. Récupérer les outputs (le déploiement est OK)
outputs_json = az(
    f"az deployment group show --resource-group {RG} --name mailroom-base "
    f"--query properties.outputs -o json"
)
if not outputs_json or outputs_json == "null":
    raise RuntimeError(
        "Le déploiement est Succeeded mais ne renvoie aucun output. "
        "Vérifiez que infra-base.bicep contient bien les blocs `output ...`."
    )

OUTPUTS = {k: v["value"] for k, v in json.loads(outputs_json).items()}

print("\nOutputs du déploiement infra-base :\n")
for k, v in OUTPUTS.items():
    print(f"  {k:30s} = {v}")

# Et on les exporte en env pour les cellules shell qui suivent
for k, v in OUTPUTS.items():
    os.environ[k.upper()] = str(v)

## 5. Créer le projet Foundry

Un **compte Foundry** (= ressource Azure de type `AIServices`) peut héberger plusieurs **projets**. Un projet, c'est ton workspace : il a son propre endpoint, ses propres déploiements de modèles, ses propres agents.

Après cette étape, tu pourras ouvrir [https://ai.azure.com](https://ai.azure.com) et voir ton projet `mailroom-project` avec le modèle `gpt-5-mini` déjà déployé (le modèle, lui, est déployé par Bicep dans la passe 1).

On crée le projet en **REST direct** parce que l'API `accounts/projects` Bicep est en preview et change tout le temps. Plus stable comme ça.

In [ ]:
# (1) On active le "custom subdomain" sur la ressource Foundry. C'est obligatoire
#     pour que les appels Entra ID (token-based auth) fonctionnent.
!az cognitiveservices account update \
    --name $FOUNDRY_NAME --resource-group $RG \
    --custom-domain $FOUNDRY_NAME \
    --output none

# (2) Création du projet Foundry via REST direct.
import json as _json
_sub = az("az account show --query id -o tsv")
_uri = (
    f"https://management.azure.com/subscriptions/{_sub}"

    f"/resourceGroups/{RG}/providers/Microsoft.CognitiveServices"!az rest --method PUT --uri "{_uri}" --body "@project-body.json" --query "{{name:name,state:properties.provisioningState}}" -o table

    f"/accounts/{FOUNDRY_NAME}/projects/{FOUNDRY_PROJECT}?api-version=2025-04-01-preview"    f.write(_json.dumps(_body))

)with open("project-body.json", "w", encoding="utf-8") as f:
_body = {"location": LOCATION, "identity": {"type": "SystemAssigned"}, "properties": {}}

## 6. Builder + pousser les images Docker

Deux images à construire :
- `mailroom-web:v1` — Next.js (~3-4 min)
- `mailroom-worker:v1` — Python (~2 min)

> 💡 ACR Tasks utilise le builder Docker **legacy** (pas BuildKit) — nos Dockerfiles sont déjà compatibles (pas de `RUN --mount=type=cache`).

On utilise **`az acr build`** qui envoie le code source à ACR et **builde dans le cloud**. Avantages :

- Le build tourne sur Linux, donc pas de souci Windows-vs-Linux.

- Tu n'as **pas besoin de Docker Desktop actif** localement.- Pas de transfert de gros layer Docker à l'upload.

In [ ]:
# Build de l'image web (Next.js). ~3-5 min.
# --no-logs : évite un bug d'encodage Windows quand Next.js affiche le triangle ▲.
!az acr build \
    --registry $ACR_NAME \
    --image mailroom-web:v1 \
    --file "{WEB_DIR}/Dockerfile" \

    --no-logs \    "{WEB_DIR}"

In [ ]:
# Build de l'image worker (Python). ~2 min.
!az acr build \
    --registry $ACR_NAME \
    --image mailroom-worker:v1 \
    --file "{WORKER_DIR}/Dockerfile" \
    --no-logs \
    "{WORKER_DIR}"

In [ ]:
# Vérifier que les 2 images sont bien dans l'ACR.
# On doit voir mailroom-web et mailroom-worker, chacun avec un tag v1.
!az acr repository list --name $ACR_NAME --output table
print()
!az acr repository show-tags --name $ACR_NAME --repository mailroom-web -o table
!az acr repository show-tags --name $ACR_NAME --repository mailroom-worker -o table

## 7. Déployer les apps (Bicep #2)

Maintenant que les images sont dans l'ACR, on peut déployer les Container Apps qui vont les exécuter.

[infra/infra-apps.bicep](infra/infra-apps.bicep) crée :

- **`web-stage-<id>`** : Container App (Next.js), ingress public HTTPS sur port 3000, 1–3 replicas auto-scalés.
- **`worker-stage-<id>`** : Container App **Job** event-driven via KEDA (azure-queue scaler). Le worker tourne uniquement quand il y a des messages dans la queue.
- **2 User-Assigned Managed Identities** (UAMI) — une par app, avec tous les rôles nécessaires déjà attribués.

> 📝 **Pourquoi UAMI et pas SystemAssigned ?** Parce que les rôles `AcrPull` doivent exister AVANT que le container ne démarre (sinon il ne peut pas pull l'image et timeout). Avec une UAMI, on crée l'identité + on l'autorise AVANT de créer le container.

⏱️ **~1 min**.

In [ ]:
# On reprend la région ACA depuis les outputs de la passe 1
# (sécurité si on a basculé de région entre temps).
ACA_LOCATION_USED = OUTPUTS.get("acaEnvLocation", ACA_LOCATION)
os.environ["ACA_LOCATION_USED"] = ACA_LOCATION_USED
print(f"ACA Env est dans : {ACA_LOCATION_USED}")

!az deployment group create \
    --name mailroom-apps \
    --resource-group $RG \
    --template-file "{INFRA_DIR}/infra-apps.bicep" \
    --parameters userId=$USER_ID acaLocation=$ACA_LOCATION_USED webImageTag=v1 workerImageTag=v1 \
    --output table

In [ ]:
# Récupère l'URL publique de la web app (générée par ACA).
FQDN = subprocess.check_output(
    f"az containerapp show -n {WEB_APP} -g {RG} --query properties.configuration.ingress.fqdn -o tsv",
    shell=True,
).decode().strip()
print(f"🌍 Web app : https://{FQDN}")
print(f"   /admin/inbox    -> page d'upload")
print(f"   /admin/clients  -> liste clients")
print(f"   /admin/storage  -> arborescence blob")

## 7.5 RBAC data-plane Cosmos DB

Azure RBAC gère les permissions de **gestion** de Cosmos (créer/supprimer un compte), mais **pas** les accès aux données. Pour lire/écrire des items, Cosmos a son propre système de RBAC é part : les **`sqlRoleAssignments`**.

Les UAMIs `web` et `worker` ont besoin du rôle **`Cosmos DB Built-in Data Contributor`** (id `00000000-0000-0000-0000-000000000002`) pour faire des `upsert` et des `query`.

Cette étape est aussi dans le Bicep (`infra-apps.bicep`), mais on la répète ici par sécurité (et pour expliquer le pourquoi).

In [ ]:
COSMOS_DATA_CONTRIB = "00000000-0000-0000-0000-000000000002"
COSMOS_ID = az(f"az cosmosdb show -n {COSMOS_NAME} -g {RG} --query id -o tsv")

for name in (f"uami-web-{USER_ID}", f"uami-worker-{USER_ID}"):
    principal = az(f"az identity show -n {name} -g {RG} --query principalId -o tsv")
    print(f"Granting Cosmos Data Contributor → {name} ({principal[:8]}…)")
    !az cosmosdb sql role assignment create \
        --account-name $COSMOS_NAME --resource-group $RG \
        --scope "{COSMOS_ID}" \
        --principal-id "{principal}" \
        --role-definition-id "{COSMOS_DATA_CONTRIB}" \
        --output none 2>nul
print("✓ Done.")

## 8. Test de bout en bout

Ouvre l'URL ci-dessus dans ton navigateur, va sur **`/admin/inbox`** et upload un PDF. Le flux complet est :

```
1. Browser   --(POST /api/upload)-->   web (Next.js)
2. web       --(PUT blob)----------->   Storage Blob (_inbox/<uuid>.pdf)
3. web       --(enqueue)------------>   Storage Queue (doc-to-classify)
4. KEDA      --(poll 30s)----------->   déclenche une exécution du Job worker
5. worker    --(download)----------->   Storage Blob (lit le PDF en bytes)
6. worker    --(prebuilt-layout)---->   Document Intelligence (OCR)
7. worker    --(chat.completions)--->   Foundry gpt-5-mini (classification structurée)
8. worker    --(upsert)------------->   Cosmos DB (documents container)
```

Durée attendue : ~20-25 s par document.

La cellule suivante vérifie qu'au moins une exécution du job a eu lieu après ton upload.

In [ ]:
# Liste les dernières exécutions du job worker.
# Après un upload, tu devrais voir une nouvelle ligne en "Succeeded" sous ~25 s.
!az containerapp job execution list \
    --name $WORKER_JOB --resource-group $RG \
    --query "[].{{name:name, status:properties.status, start:properties.startTime}}" \
    -o table

In [ ]:
# Logs applicatifs du worker via Log Analytics (KQL).
LAW_ID = az(f"az monitor log-analytics workspace show -n law-stage-{USER_ID} -g {RG} --query customerId -o tsv")
print(f"Workspace ID: {LAW_ID}\n")

QUERY = (
    "ContainerAppConsoleLogs_CL "
    "| where TimeGenerated > ago(10m) "
    "| where ContainerName_s == 'worker' "
    "| project TimeGenerated, Log_s "
    "| order by TimeGenerated desc "
    "| take 20"
)
!az monitor log-analytics query --workspace "{LAW_ID}" --analytics-query "{QUERY}" -o tsv

## 9. 🧹 Nettoyage ciblé

⚠️ **Ne supprime JAMAIS le Resource Group lui-même** (il appartient à ton équipe).

Pour tout détruire sauf le RG, **décommente les lignes qui t'intéressent** dans la cellule suivante.

💡 **Astuce coûts** : si tu veux juste "mettre en pause" sans tout supprimer, scale le `web` à 0 replica :

```bash
az containerapp update -n web-stage-<id> -g rg-stage-<id> --min-replicas 0 --max-replicas 0
```

Le worker, lui, est déjà à 0 quand la queue est vide (KEDA).

In [ ]:
# Décommentez pour exécuter

# # Apps + Job (libère les coûts ACA)
# !az containerapp delete --name $WEB_APP --resource-group $RG --yes
# !az containerapp job delete --name $WORKER_JOB --resource-group $RG --yes
# !az containerapp env delete --name $ACA_ENV --resource-group $RG --yes

# # Foundry (projet, puis ressource)
# !az cognitiveservices account project delete --name $FOUNDRY_NAME --resource-group $RG --project-name $FOUNDRY_PROJECT
# !az cognitiveservices account delete --name $FOUNDRY_NAME --resource-group $RG

# # Cosmos
# !az cosmosdb delete --name $COSMOS_NAME --resource-group $RG --yes

# # Storage + ACR + observabilité
# !az storage account delete --name $STORAGE_NAME --resource-group $RG --yes
# !az acr delete --name $ACR_NAME --resource-group $RG --yes
# !az monitor app-insights component delete --app appi-stage-$USER_ID --resource-group $RG
# !az monitor log-analytics workspace delete --workspace-name law-stage-$USER_ID --resource-group $RG --yes --force true

print(f"⚠️  Resource Group préservé : {RG}")
print("    Décommentez les lignes ci-dessus pour supprimer les ressources individuellement.")

## Récap

Tu as :
- Déployé toute l'infra en **2 passes Bicep** (base puis apps).
- Créé un **projet Foundry** + déployé le modèle `gpt-5-mini`.
- Buildé et poussé **2 images Docker** dans ton ACR managed.
- Déployé un **Container App** (`web`, ingress public) et un **Container App Job** event-driven KEDA (`worker`).
- Câblé toutes les **Managed Identities + RBAC** (Azure RBAC pour Storage/ACR/Foundry, RBAC natif Cosmos pour le data plane).
- Testé le pipeline upload → OCR → LLM → Cosmos de bout en bout.

## Schema architecture

Ouvre [docs/architecture.drawio](docs/architecture.drawio) pour voir le diagramme complet (extension VS Code *Draw.io Integration*, ou import sur https://app.diagrams.net).

## Pour aller plus loin

- **Réseau** : remplacer le public-access par VNet + Private Endpoints + App Gateway. Cf. ADR future dans `DESIGN.md`.

- **Auth utilisateurs** : ajouter Entra External ID (clients) + Entra ID (admins). Aujourd'hui, l'admin UI est en accès libre — à NE PAS faire en prod.- **Évaluations Foundry** : voir notebook 06 du parcours.

- **CRUD clients** : finaliser la page `/admin/clients`.- **CI/CD** : pipeline GitHub Actions qui rebuilde + redeploy sur push.